# Apache Beam: Building Data Pipelines
Apache Beam is an open-source unified programming model used to define both **batch** and **streaming data processing pipelines**. In this notebook, we'll walk through a series of steps to build simple data pipelines using **Apache Beam**. 

We'll start with a local pipeline to understand basic transformations and then move on working with Google Cloud Storage (GCS) for output.

## Step 1: Installing Apache Beam
We begin by installing **Apache Beam**.

```!pip install apache-beam```

## Simple Pipeline - Square Numbers 

In this first example, we create a basic pipeline that reads a list of numbers, squares them, and prints the results. Let's dive into it!

In [7]:
# Import the necessary libraries
import apache_beam as beam

# Initialize the pipeline
p1 = beam.Pipeline()

# Define input data
input_data = [1, 2, 3, 4, 5]

# Apply transformations: 
# 1. 'Create' to load input_data into the pipeline
# 2. 'Square' to square each number
# 3. 'Print' to print the squared numbers
result = (
    p1
    | 'Create' >> beam.Create(input_data)
    | 'Square' >> beam.Map(lambda x: x * x)
    | 'Print' >> beam.Map(print)
)

# Run the pipeline
p1.run()


1
4
9
16
25


## Building a Data Pipeline for Flight Data

In this example, we will work with a CSV dataset containing flight information. The goal is to:

- Filter flights with delay times greater than 0.
- Sum delays by airline.
- Count flights for each airline.

We'll read the data from a local CSV file and process it using Apache Beam transformations.

In [10]:
import apache_beam as beam

p1 = beam.Pipeline()

class Filter(beam.DoFn):
  def process(self,record):
    if int(record[8]) > 0:
      return [record]

Delayed_time = (
p1
  | "Import Data time" >> beam.io.ReadFromText(r"/Users/apple/Documents/GitHub/gcp-cook-book/data/flights_sample.csv", skip_header_lines = 1)
  | "Split by comma time" >> beam.Map(lambda record: record.split(','))
  | "Filter Delays time" >> beam.ParDo(Filter()) # Filter delays
  | "Create a key-value time" >> beam.Map(lambda record: (record[4],int(record[8])))
  | "Sum by key time" >> beam.CombinePerKey(sum) # Sum delay times by airline
)

Delayed_num = (
    p1
    | "Import Data" >> beam.io.ReadFromText(r"/Users/apple/Documents/GitHub/gcp-cook-book/data/flights_sample.csv", skip_header_lines = 1)
    | "Split by comma" >> beam.Map(lambda record: record.split(','))
    | "Filter Delays" >> beam.ParDo(Filter()) # Filter delays
    | "Create a key-value" >> beam.Map(lambda record: (record[4],int(record[8]))) # Airline -> delay
    | "Count by key" >> beam.combiners.Count.PerKey() # Count flights per airline
)

Delay_table = (
    {'Delayed_num':Delayed_num,'Delayed_time':Delayed_time} 
    | beam.CoGroupByKey() # Group by airline
    | beam.Map(print)
)

p1.run()

('LAX', {'Delayed_num': [4], 'Delayed_time': [92]})
('HNL', {'Delayed_num': [1], 'Delayed_time': [15]})
('DFW', {'Delayed_num': [1], 'Delayed_time': [95]})
('OGG', {'Delayed_num': [1], 'Delayed_time': [138]})
('JFK', {'Delayed_num': [4], 'Delayed_time': [220]})


## Building a Data Pipeline for Flight Data and store in GCS

In this example, we will work with a CSV dataset containing flight information. The goal is to:

- Filter flights with delay times greater than 0.
- Sum delays by airline.
- Count flights for each airline.

We'll read the data from a local CSV file and process it using Apache Beam transformations.

We’ll integrate load data to Google Cloud Storage. This involves setting up authentication with Google Cloud credentials and modifying the pipeline to store output in Google Cloud Storage (GCS).

In [11]:
import apache_beam as beam
import os
from datetime import datetime

# Set Google Cloud credentials (use your service account file)
serviceAccount = r"/Users/apple/Documents/GitHub/gcp-cook-book/service-account/spring-base-455810-r1-6c569e4617ef.json" #path to your service account file locally
os.environ["GOOGLE_APPLICATION_CREDENTIALS"]= serviceAccount

# Initialize the pipeline
p1 = beam.Pipeline()

# Get the current date for partitioning
current_date = datetime.utcnow()
year = current_date.year
month = f"{current_date.month:02d}"  # zero-padded
day = f"{current_date.day:02d}"      # zero-padded

# Filter function to keep records with delay times > 0
class Filter(beam.DoFn):
  def process(self,record):
    if int(record[8]) > 0:
      return [record]

# Apply transformations as before
Delayed_time = (
p1
  | "Import Data time" >> beam.io.ReadFromText(r"/Users/apple/Documents/GitHub/gcp-cook-book/data/flights_sample.csv", skip_header_lines = 1)
  | "Split by comma time" >> beam.Map(lambda record: record.split(','))
  | "Filter Delays time" >> beam.ParDo(Filter())
  | "Create a key-value time" >> beam.Map(lambda record: (record[4],int(record[8])))
  | "Sum by key time" >> beam.CombinePerKey(sum)
)

Delayed_num = (
    p1
    | "Import Data" >> beam.io.ReadFromText(r"/Users/apple/Documents/GitHub/gcp-cook-book/data/flights_sample.csv", skip_header_lines = 1)
    | "Split by comma" >> beam.Map(lambda record: record.split(','))
    | "Filter Delays" >> beam.ParDo(Filter())
    | "Create a key-value" >> beam.Map(lambda record: (record[4],int(record[8])))
    | "Count by key" >> beam.combiners.Count.PerKey()
)

# Save results to Google Cloud Storage
Delay_table = (
    {'Delayed_num':Delayed_num,'Delayed_time':Delayed_time} 
    | "Group By" >> beam.CoGroupByKey()
    | "Save to GCS" >> beam.io.WriteToText(f"gs://df_temp_pipeline/{year}/{month}/{day}/flights_output.csv")
)

p1.run()


# Output

![GCS Flight Output](images/GCS_flight_output.png)
![Flight Output CSV](images/flight_output_csv.png)